# CRM · SEC EDGAR service on Colab

Runs `services/edgar` here and exposes it through a Cloudflare quick tunnel. Fill the two values in the next cell, run every cell, then paste the printed `EDGAR_URL` and `EDGAR_SECRET` into the CRM's environment. The service lives as long as this notebook runs.

In [ ]:
IDENTITY = "Jane Doe jane@example.com"  # the SEC asks every automated client for a name and a real email
REPO = "https://github.com/teknewmcc26/crm"  # the CRM repository that holds services/edgar


In [ ]:
import secrets, subprocess, sys

SECRET = secrets.token_hex(24)
subprocess.run(["git", "clone", "--depth", "1", REPO, "crm"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "crm/services/edgar/requirements.txt"], check=True)
subprocess.run(["bash", "-c", "curl -sSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"], check=True)
print("installed")


In [ ]:
import os, subprocess, time

env = dict(os.environ, EDGAR_IDENTITY=IDENTITY, EDGAR_SECRET=SECRET, EDGAR_PORT="2100", EDGAR_DATA_DIR="/content/edgar-cache")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "edgar_service.app:app", "--host", "0.0.0.0", "--port", "2100"], cwd="crm/services/edgar", env=env)
time.sleep(5)
print(subprocess.run(["curl", "-s", "http://127.0.0.1:2100/health"], capture_output=True, text=True).stdout)


In [ ]:
import re, subprocess, time

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:2100", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        url = found.group(0)
        break
    time.sleep(0.5)
print("EDGAR_URL=" + (url or "<tunnel did not start; rerun this cell>"))
print("EDGAR_SECRET=" + SECRET)


Set both values in the CRM (Vercel → crm-agent → Environment Variables, or the local `.env`) and redeploy or restart the agent. `sec_search_companies` then answers from here.